# Econ 148 Final Project - Data Inspection

Track A, Topic 4: Intergenerational Mobility Across US Counties.
Team: Paul (data engineering), Kabir, Emily.

## Project scope

Our research question is: **what interactions between Chetty's five correlates of
intergenerational mobility (segregation, income inequality, school quality, social
capital, family structure) does a random forest capture that OLS misses?** To
answer it we need a county-level feature matrix and outcome vector, both built
from Opportunity Insights data.

This first notebook is strictly a data-inspection pass: we download the three
source files, load each into a pandas DataFrame, and verify that their FIPS
county identifiers line up so we can later merge features onto outcomes.

The three datasets loaded below are:

1. **Chetty et al. 2014 "Where is the Land of Opportunity?"** - online data tables
   Excel workbook. Online Data Table 3 is the county-level table of mobility
   statistics and income-distribution covariates that we load here; Online Data
   Tables 5 and 8 (commuting-zone correlates) are available in the same workbook
   for later use.
2. **Opportunity Atlas 2018 tract-level outcomes** - the "simple" variant of the
   Atlas tract file (~33 MB). Each row is a census tract; the `state`, `county`,
   and `tract` integer columns combine to a standard 11-digit tract FIPS, and
   rolling up to county FIPS is a single groupby away.
3. **Atlas county-level outcomes rollup** - a ready-made county-level version of
   the same Atlas outcomes. We use this as the day-1 outcome variable so that
   the modelling pipeline can start before tract-to-county aggregation is built.

## Imports and path setup

We pull in the standard scientific-Python stack plus our own `src.data_loader`
module. The path-discovery loop below walks up from the notebook's working
directory to find the project root (marked by the `.git` folder), so that
`from src.data_loader import ...` works whether the notebook is launched from
the project root or from the `notebooks/` subdirectory.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'src' / 'data_loader.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from src.data_loader import (
    get_chetty_2014,
    get_opportunity_atlas_tract,
    get_county_outcomes,
)

print('pandas', pd.__version__, '  numpy', np.__version__)

## Download the three source files

Each `get_*` function:

- checks `data/raw/<filename>` and skips the download if a cached copy exists;
- otherwise tries each URL in its internal fallback list in order, logging every
  attempt (`try:` / `fail:` / `saved:`) so we can see which source actually served
  the file;
- prints the resulting file size on success.

The fallback lists in `src/data_loader.py` include the URLs originally named in
the project brief plus the `equality-of-opportunity.org` and `www2.census.gov`
mirrors, because several of the `opportunityinsights.org/wp-content/uploads/...`
paths were reorganized between 2018 and 2026 and now return 404 or 403.

In [ ]:
chetty_path = get_chetty_2014()
tract_path  = get_opportunity_atlas_tract()
county_path = get_county_outcomes()

## Dataset 1 of 3: Chetty et al. 2014 descriptive tables

**What we expect.** A multi-sheet Excel workbook. Online Data Table 3 is a
county-level table (~3,141 US counties) with columns including `County FIPS
Code`, `County Name`, `Commuting Zone ID`, `Rank-Rank Slope`, `Absolute Upward
Mobility`, `Gini`, `Teenage Birth Rate`, and various parent/child income
percentiles. We load this sheet as the first-pass feature set.

Online Data Tables 5-8 carry CZ-level data, including the famous five correlates
of mobility; we leave those for a later notebook once the team decides whether
to model at CZ or county resolution.

### 1a. Load the workbook

The header for Online Data Table 3 sits on row 30 of the raw sheet (0-indexed
29), preceded by title and explanatory-note rows, and followed by a short row of
column-number labels (`(1)`, `(2)`, ...) that we drop before the real data
begins. We also coerce numeric columns to numeric dtypes (pandas otherwise
leaves them as `object` because of the dropped label row).

In [ ]:
chetty_df = pd.read_excel(
    chetty_path,
    sheet_name='Online Data Table 3',
    skiprows=29,
    header=0,
)
chetty_df = chetty_df.iloc[1:].reset_index(drop=True)

_text_cols = {'County Name', 'Commuting Zone Name ', 'State'}
for _c in chetty_df.columns:
    if _c not in _text_cols:
        chetty_df[_c] = pd.to_numeric(chetty_df[_c], errors='coerce')

chetty_df.head(3)

### 1b. Shape, info, and head

A quick structural inspection: row and column counts, per-column dtypes and
non-null counts, and the first five records.

In [ ]:
print('shape:', chetty_df.shape)
print()
chetty_df.info()
print()
chetty_df.head()

### 1c. FIPS column in Chetty 2014

Scan the columns for anything that looks like a county identifier (`FIPS`,
`cty`, `county`, `statecounty`, or a value that is a 2-digit state + 3-digit
county pattern). Report its dtype, a sample of five non-null values, and the
percentage missing.

In [ ]:
_candidates = [c for c in chetty_df.columns
               if any(k in c.lower() for k in ('fips', 'cty', 'county', 'statecounty'))]
print('candidate FIPS-like columns:', _candidates)

chetty_fips_col = 'County FIPS Code'
_values = chetty_df[chetty_fips_col]

print()
print('chosen column:', chetty_fips_col)
print('dtype:        ', _values.dtype)
print('sample of 5:  ', _values.dropna().sample(5, random_state=0).astype(int).tolist())
print('pct missing:   {:.3%}'.format(_values.isna().mean()))

## Dataset 2 of 3: Opportunity Atlas tract-level outcomes

**What we expect.** ~73,000 rows (one per US census tract) and ~64 columns of
pooled and race/gender-disaggregated mobility estimates. The key outcome column
is `kfr_pooled_pooled_p25` (mean child rank at parent p25). Tract identification
is split across three integer columns: `state` (1-56 FIPS state), `county` (the
3-digit county-within-state FIPS), and `tract` (the 6-digit tract-within-county
FIPS).

### 2a. Load the tract CSV

In [ ]:
tract_df = pd.read_csv(tract_path)
tract_df.head(3)

### 2b. Shape, info, and head

With ~64 columns we pass `verbose=True, show_counts=True` so `info()` prints
per-column non-null counts instead of a summary.

In [ ]:
print('shape:', tract_df.shape)
print()
tract_df.info(verbose=True, show_counts=True)
print()
tract_df.head()

### 2c. FIPS column in the tract file

There is no single FIPS column: identifiers are split across `state`, `county`,
and `tract`. We derive a standard 5-digit county FIPS as `state * 1000 +
county`, which matches the packed integer representation used by Chetty 2014
Online Data Table 3 (e.g. Alabama / Autauga = 1001). The 11-digit tract FIPS
(`state * 1e9 + county * 1e6 + tract`) will become useful when we aggregate
tracts to counties in a later notebook.

In [ ]:
tract_df['county_fips'] = tract_df['state'] * 1000 + tract_df['county']

_candidates = [c for c in tract_df.columns
               if any(k in c.lower() for k in ('fips', 'cty', 'county', 'statecounty'))]
print('candidate FIPS-like columns:', _candidates)

print()
print('chosen column: county_fips (derived = state*1000 + county)')
print('dtype:        ', tract_df['county_fips'].dtype)
print('sample of 5:  ', tract_df['county_fips'].dropna().sample(5, random_state=0).astype(int).tolist())
print('unique counties represented:', tract_df['county_fips'].nunique())
print('pct missing:   {:.3%}'.format(tract_df['county_fips'].isna().mean()))

## Dataset 3 of 3: Atlas county-level outcomes (day-1 outcome)

**What we expect.** One row per county (~3,219 rows) with the same outcome
columns as the tract file but pre-aggregated. This is the day-1 outcome
variable: it lets us start fitting models without implementing the
tract-to-county aggregation first. The identifier columns are `state` (2-digit
FIPS) and `county` (3-digit FIPS).

### 3a. Load the county CSV

In [ ]:
county_df = pd.read_csv(county_path)
county_df.head(3)

### 3b. Shape, info, and head

In [ ]:
print('shape:', county_df.shape)
print()
county_df.info(verbose=True, show_counts=True)
print()
county_df.head()

### 3c. FIPS column in the county file

Same `state`/`county` integer split as the tract file; we build the 5-digit key
`county_fips = state * 1000 + county` and report missingness for both the key
and the main outcome `kfr_pooled_pooled_p25`.

In [ ]:
county_df['county_fips'] = county_df['state'] * 1000 + county_df['county']

_candidates = [c for c in county_df.columns
               if any(k in c.lower() for k in ('fips', 'cty', 'county', 'statecounty'))]
print('candidate FIPS-like columns:', _candidates)

print()
print('chosen column: county_fips (derived = state*1000 + county)')
print('dtype:        ', county_df['county_fips'].dtype)
print('sample of 5:  ', county_df['county_fips'].dropna().sample(5, random_state=0).astype(int).tolist())
print('pct missing county_fips:            {:.3%}'.format(county_df['county_fips'].isna().mean()))
print('pct missing kfr_pooled_pooled_p25:  {:.3%}'.format(county_df['kfr_pooled_pooled_p25'].isna().mean()))

## Cross-dataset FIPS compatibility check

The three datasets will be merged on 5-digit county FIPS. Below we compute the
pairwise and three-way intersections of the FIPS key sets and count any counties
that appear in some frames but not others. A clean merge requires every county
we care about to be present in all three.

In [ ]:
chetty_fips_set = set(chetty_df['County FIPS Code'].dropna().astype(int).tolist())
tract_fips_set  = set(tract_df['county_fips'].dropna().astype(int).unique().tolist())
county_fips_set = set(county_df['county_fips'].dropna().astype(int).tolist())

print('Unique counties per dataset:')
print(f'  Chetty 2014 Online Data Table 3:   {len(chetty_fips_set):>5}')
print(f'  Atlas tract file (rolled to cty):  {len(tract_fips_set):>5}')
print(f'  Atlas county outcomes rollup:      {len(county_fips_set):>5}')

all_three    = chetty_fips_set & tract_fips_set & county_fips_set
chetty_only  = chetty_fips_set - (tract_fips_set | county_fips_set)
outcome_only = county_fips_set - chetty_fips_set
tract_only   = tract_fips_set - chetty_fips_set

print()
print(f'Counties in ALL three:                 {len(all_three):>5}')
print(f'Counties in Chetty only (no outcome):  {len(chetty_only):>5}')
print(f'Counties in Atlas but not Chetty:      {len(outcome_only):>5}')
print(f'Counties in tract but not Chetty:      {len(tract_only):>5}')

## Summary for Leo (Progress Report input)

A concise three-sentence data-engineering summary to paste into the Progress
Report. Regenerated from live data each time the notebook is run, so numbers
stay in sync with the frames above.

In [ ]:
chetty_na_pct = chetty_df['County FIPS Code'].isna().mean()
outcome_na_pct = county_df['kfr_pooled_pooled_p25'].isna().mean()
shared = len(chetty_fips_set & county_fips_set)
only_chetty = len(chetty_fips_set - county_fips_set)
only_outcome = len(county_fips_set - chetty_fips_set)

summary = (
    f"(1) All three Opportunity Insights datasets downloaded and loaded: Chetty 2014 "
    f"Online Data Table 3 (county-level mobility and income covariates), the "
    f"Opportunity Atlas tract-level outcomes (simple variant, ~33 MB), and the "
    f"Atlas county-level outcome rollup. "
    f"(2) Row x column counts are {chetty_df.shape} for Chetty 2014, "
    f"{tract_df.shape} for the tract file, and {county_df.shape} for the county "
    f"outcomes; they share a 5-digit county FIPS key on which {shared} counties "
    f"intersect between the Chetty feature file and the outcome file. "
    f"(3) Flagged issues: Chetty FIPS is {chetty_na_pct:.2%} missing and the main "
    f"outcome kfr_pooled_pooled_p25 is {outcome_na_pct:.2%} missing, with "
    f"{only_chetty} Chetty counties lacking an Atlas outcome and {only_outcome} "
    f"Atlas counties lacking Chetty covariates; we will decide between dropping or "
    f"imputing these in the next step."
)
print(summary)

## CZ-level covariates merge preview

Purpose: preview the feature matrix that the OLS specification will use. Our
features are Chetty's "famous five" CZ-level covariates (racial and income
segregation, income inequality, school quality, social capital, family
structure) and related CZ characteristics, and the outcome is
`kfr_pooled_pooled_p25` on the county frame. This section loads the covariates,
loads the county-to-CZ crosswalk, performs the left merge, and reports match
rate and the final feature column list.

Note on sheet selection in the 2014 workbook: the project brief referred to
"Online Data Table 5" for the famous five covariates. In the published
workbook Online Data Table 5 actually holds CZ-level mobility *estimates*
(rank-rank slope, absolute upward mobility, college attendance rates at parent
p25), while the CZ-level *covariates* used in the correlation analysis of
Chetty et al. (2014) Section V live in Online Data Table 8 ("Commuting Zone
Characteristics"). We load Table 8 here because the merge preview requires
covariates. If the team prefers to also pull Table 5's mobility estimates at
CZ level for a secondary regression, the loader can be reused with
`sheet_name='Online Data Table 5'` in a future notebook.

### 4a. Load the CZ-level covariates (Online Data Table 8)

Table 8's header sits on row 7 of the raw sheet (0-indexed 6); row 8 is a short
column-number row that we drop before the real data starts. Numeric columns
are coerced with `pd.to_numeric(errors='coerce')` because the dropped row
forces pandas to infer `object` dtype for the entire column.

In [ ]:
cz_cov_df = pd.read_excel(
    chetty_path,
    sheet_name='Online Data Table 8',
    skiprows=6,
    header=0,
)
cz_cov_df = cz_cov_df.iloc[1:].reset_index(drop=True)

_cz_cov_text_cols = {'CZ Name', 'State'}
for _c in cz_cov_df.columns:
    if _c not in _cz_cov_text_cols:
        cz_cov_df[_c] = pd.to_numeric(cz_cov_df[_c], errors='coerce')

print('CZ covariates shape:', cz_cov_df.shape)
print('first 8 columns:    ', list(cz_cov_df.columns)[:8])
cz_cov_df.head(3)

### 4b. Download and load the county-to-CZ crosswalk

The crosswalk maps each 5-digit county FIPS (`cty`) to its commuting-zone id
(`cz`). We use it as the authoritative county-to-CZ mapping for the merge
(rather than trusting the `cz` column already present on the county outcomes
file), so that the pipeline also works for datasets that arrive without a
pre-attached CZ column.

In [ ]:
from src.data_loader import download_cz_crosswalk

crosswalk_path = download_cz_crosswalk()
crosswalk_df = pd.read_csv(crosswalk_path)
print('crosswalk shape:', crosswalk_df.shape)
print('cols:           ', list(crosswalk_df.columns))
crosswalk_df.head(3)

### 4c. Left-merge CZ covariates onto the county frame

Two-step merge so the provenance of every column is explicit:

1. Attach the crosswalk's `cz` onto `county_df` using `county_fips` as the key.
2. Attach `cz_cov_df` (Online Data Table 8) onto that intermediate frame using
   `cz` as the key.

Left join at each step so every original county row is preserved; unmatched
rows will carry `NaN` in the covariate columns and will surface in the match-
rate check below.

In [ ]:
_cw_narrow = crosswalk_df[['cty', 'cz']].rename(columns={'cty': 'county_fips', 'cz': 'cz_from_crosswalk'})
county_with_cz = county_df.merge(_cw_narrow, on='county_fips', how='left')

_cz_cov_for_merge = cz_cov_df.rename(columns={'CZ': 'cz_from_crosswalk'})
merged_df = county_with_cz.merge(_cz_cov_for_merge, on='cz_from_crosswalk', how='left')

print('county_df shape before merge:          ', county_df.shape)
print('+ crosswalk cz attached:                ', county_with_cz.shape)
print('+ Table 8 CZ covariates left-joined:    ', merged_df.shape)

### 4d. Match rate and merged feature columns

We quantify the merge quality two ways. First, the share of county rows that
received a non-null CZ id from the crosswalk. Second, the share of county rows
whose first CZ-level covariate (`Racial Segregation`, a proxy for "all CZ
covariates merged on successfully") is non-null after the join. The second
number is the one to cite when talking about how many counties the OLS spec
will actually see.

In [ ]:
cz_match = merged_df['cz_from_crosswalk'].notna().mean()
cov_match = merged_df['Racial Segregation'].notna().mean()

_cov_feature_cols = [c for c in cz_cov_df.columns if c not in ('CZ', 'CZ Name', 'State')]

print(f'Rows in merged county frame:                 {len(merged_df)}')
print(f'County rows with CZ id from crosswalk:       {merged_df["cz_from_crosswalk"].notna().sum()}  '
      f'({cz_match:.2%})')
print(f'County rows with non-null CZ covariates:     {merged_df["Racial Segregation"].notna().sum()}  '
      f'({cov_match:.2%})')
print()
print(f'Merged CZ-covariate feature columns ({len(_cov_feature_cols)}):')
for _c in _cov_feature_cols:
    print(f'  - {_c}')